In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("listings_clean.csv")
print(df.shape)

# columns that are text (not numbers) still need converting
text_cols = df.select_dtypes(exclude="number").columns.tolist()
print(text_cols)
for c in text_cols:
    print(c, df[c].nunique(), df[c].unique()[:5])

(58895, 51)
['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'property_type', 'room_type', 'bathrooms_text', 'amenities', 'has_availability']
host_is_superhost 2 ['f' 't']
host_has_profile_pic 2 ['t' 'f']
host_identity_verified 2 ['t' 'f']
neighbourhood_cleansed 33 ['Lambeth' 'Islington' 'Kensington and Chelsea' 'Wandsworth'
 'Richmond upon Thames']
property_type 84 ['Entire rental unit' 'Private room in rental unit' 'Entire townhouse'
 'Private room in home' 'Entire serviced apartment']
room_type 4 ['Entire home/apt' 'Private room' 'Hotel room' 'Shared room']
bathrooms_text 40 ['1 bath' '1 shared bath' '1.5 baths' '2 baths' '1 private bath']
amenities 53493 ['["40 inch HDTV with Netflix, standard cable", "Wine glasses", "Shower gel", "Other stainless steel gas stove", "Essentials", "Laundromat nearby", "Kitchenette", "Microwave", "Smoke alarm", "Bathtub", "Luggage dropoff allowed", "Coffee maker", "Hair dryer", "Body soap", "Paid street

In [2]:
# true/false text -> 1/0
for c in ["host_is_superhost", "host_has_profile_pic",
          "host_identity_verified", "has_availability"]:
    df[c] = df[c].map({"t": 1, "f": 0})

# bathrooms_text like "1.5 baths" -> number, plus a "shared" flag
df["bathrooms_num"] = df["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float)
half = df["bathrooms_text"].str.contains("half", case=False, na=False)
df.loc[half & df["bathrooms_num"].isna(), "bathrooms_num"] = 0.5
df["bath_shared"] = df["bathrooms_text"].str.contains("shared", case=False, na=False).astype(int)
df = df.drop(columns=["bathrooms_text"])

print(df[["host_is_superhost", "host_identity_verified", "bathrooms_num", "bath_shared"]].describe())
print(df["bathrooms_num"].isna().sum(), "bathroom values missing")

       host_is_superhost  host_identity_verified  bathrooms_num   bath_shared
count       58895.000000            58895.000000   58802.000000  58895.000000
mean            0.249987                0.984566       1.330541      0.176399
std             0.433009                0.123273       0.674193      0.381162
min             0.000000                0.000000       0.000000      0.000000
25%             0.000000                1.000000       1.000000      0.000000
50%             0.000000                1.000000       1.000000      0.000000
75%             0.000000                1.000000       1.500000      0.000000
max             1.000000                1.000000      30.000000      1.000000
93 bathroom values missing


In [3]:
import json

# --- property_type: keep the 10 most common, group the rest as "Other" ---
top_props = df["property_type"].value_counts().head(10).index
df["property_type"] = df["property_type"].where(df["property_type"].isin(top_props), "Other")

# --- amenities: text that looks like a list -> list -> 0/1 columns for the 20 most common ---
df["amenities"] = df["amenities"].apply(json.loads)
from collections import Counter
counts = Counter(a for lst in df["amenities"] for a in lst)
top_amen = [a for a, _ in counts.most_common(20)]
print(top_amen)

for a in top_amen:
    df["amen_" + a.lower().replace(" ", "_").replace("/", "_")] = df["amenities"].apply(lambda lst: int(a in lst))
df["amenity_count"] = df["amenities"].apply(len)
df = df.drop(columns=["amenities"])

# --- one-hot encode the three categorical columns ---
df = pd.get_dummies(df, columns=["room_type", "neighbourhood_cleansed", "property_type"], dtype=int)
print(df.shape)

['Wifi', 'Smoke alarm', 'Kitchen', 'Hot water', 'Carbon monoxide alarm', 'Iron', 'Hair dryer', 'Hangers', 'Bed linens', 'Washer', 'Dishes and silverware', 'Refrigerator', 'Essentials', 'Cooking basics', 'TV', 'Hot water kettle', 'Microwave', 'Heating', 'Shampoo', 'Toaster']
(58895, 117)


In [4]:
from sklearn.model_selection import train_test_split

y = df["price"]                 # target: nightly price in dollars
X = df.drop(columns=["price"])  # features

# 70% train, 15% validation, 15% test (fixed seed so results are repeatable)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print("train:", X_train.shape, "validation:", X_val.shape, "test:", X_test.shape)

# which columns still have gaps (in the training set)
gaps = X_train.isna().sum()
print(gaps[gaps > 0])

train: (41226, 116) validation: (8834, 116) test: (8835, 116)
bedrooms                         16
beds                           1055
minimum_nights                  217
maximum_nights                  217
minimum_minimum_nights          217
maximum_minimum_nights          217
minimum_maximum_nights          217
maximum_maximum_nights          217
has_availability                  2
review_scores_rating           8239
review_scores_accuracy         8242
review_scores_cleanliness      8242
review_scores_checkin          8242
review_scores_communication    8242
review_scores_location         8243
review_scores_value            8242
bathrooms_num                    66
dtype: int64


In [5]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)          # learn the medians from TRAINING data only

X_train_i = pd.DataFrame(imputer.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_i   = pd.DataFrame(imputer.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test_i  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)

print("gaps left:", X_train_i.isna().sum().sum(), X_val_i.isna().sum().sum(), X_test_i.isna().sum().sum())
print(X_train_i.shape)

gaps left: 0 0 0
(41226, 116)


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name}:  RMSE ${rmse:.1f}   MAE ${mae:.1f}   R2 {r2:.3f}")

# Baseline 1: predict the training median for every listing
report("Median baseline", y_val, np.full(len(y_val), y_train.median()))

# Baseline 2: linear regression on log(price), scaled features
scaler = StandardScaler().fit(X_train_i)          # scale using TRAINING data only
Xtr, Xv = scaler.transform(X_train_i), scaler.transform(X_val_i)

lin = LinearRegression().fit(Xtr, np.log1p(y_train))
pred_log = lin.predict(Xv)
report("Linear regression", y_val, np.expm1(pred_log))                 # in dollars
print("Linear regression R2 in log space:", round(r2_score(np.log1p(y_val), pred_log), 3))

Median baseline:  RMSE $166.5   MAE $112.7   R2 -0.067
Linear regression:  RMSE $104.5   MAE $61.3   R2 0.580
Linear regression R2 in log space: 0.713
